In [214]:
from datetime import datetime
from dateutil.relativedelta import relativedelta;
import yfinance  as yf
import pandas as pd
import os
import matplotlib.pyplot as plt
# Loading AAPL stock data of the last 10 year

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)


# Fecthign data
file_name="appl_10y.csv";
if(os.path.exists(file_name)):
    stock_data = pd.read_csv(file_name,index_col=0, parse_dates=True);
else:
    ticker = ["AAPL"]
    end_date = datetime.now()
    start_date = end_date -  relativedelta(years=10)
    stock_data = yf.download(ticker, start=start_date, end=end_date) 
    stock_data.columns = stock_data.columns.droplevel(1)

# -----------------------------------------------------------------------------

# Calcualte percentage change for each row
stock_data['daily_pct_change'] = stock_data['Close'].pct_change() * 100;

# Dropping reducadnat data row
stock_data = stock_data.dropna(subset=['daily_pct_change'])

# Reviewing summary of the pct chaneg column

returns = stock_data['daily_pct_change']


# plt.figure(figsize=(10, 5))

# your actual returns, as a histogram
# density=True makes the area sum to 1, so the bell curve can sit on top
# plt.hist(returns, bins=100, color='lightgreen', edgecolor='gray', density=True)

# a bell curve with the SAME mean and std as your data
# x = np.linspace(returns.min(), returns.max(), 300)
# mean = returns.mean()
# std = returns.std()
# bell = (1 / (std * np.sqrt(2 * np.pi))) * np.exp(-0.5 * ((x - mean) / std) ** 2)
# plt.plot(x, bell, color='red', linewidth=2, label='what a bell curve would look like')

# plt.title('AAPL daily returns vs a normal distribution')
# plt.xlabel('daily % change')
# plt.ylabel('density')
# plt.legend()
# plt.grid(alpha=0.3)
# plt.show()
# -----------------------------------------------------------------------------


## Feature Selection

Before building anything, I'm committing to a fixed set of features chosen from
reasoning alone. This matters: if I picked features by trying lots of them and
keeping whichever performed best, my final result would just be measuring
"how well do the features that happened to look good on this data perform on
this data?" — which proves nothing.

So: features decided first, results looked at afterwards, and no changing them
once I see how it goes.

---

### The hypothesis

A stock that has been trending over a 20-day window tends to continue in that
direction over the following 5 days — **provided the move is backed by genuine
participation** rather than being a thin, drifting move that fizzles out.

Target horizon is 5 days rather than 1, because single-day returns are close to
pure noise. Five days gives any real signal room to show up.


### Feature 1: volume_ratio_20_40

**Definition**

    average daily volume over the last 20 days
    ----------------------------------------------  
    average daily volume over the 40 days before that

Reading it:
- 1.0 -> recent trading activity matches the baseline
- 1.5 -> recent days are 50% busier than normal
- 0.6  -> recent days are quieter than normal

Needs 60 days of history in total (40 baseline + 20 recent).

**Why I chose it**

A price move backed by heavy volume means a lot of people are acting on it.
A move on thin volume is a few participants drifting the price around, and
tends to fizzle. So volume is my confirmation signal — it tells me whether the
20-day price move has real weight behind it.

**Why a 40-day baseline rather than 20**

More days averaged means a more stable estimate of "normal" volume (standard
error shrinks with the square root of sample size). The cost is that the
baseline reaches slightly further back, so it reflects marginally older
conditions. Worth it for the stability.

**What it doesn't capture**

The *shape* of the change. A single volume spike and a steady climb over 20 days
produce the same value. It tells me the recent level is elevated, not how it
got there.

**A known limitation of the hypothesis itself**

My actual belief is that volume matters *conditional on* price movement —
"a big move **with** volume behind it." That's an interaction between two
features. A linear model treats each feature independently, so it will pick up
volume's standalone effect but not the interaction. I'm accepting that for now;
an interaction term (price_change x volume_ratio) would be the fix, but I'm
keeping the feature count low to limit overfitting.

In [215]:
stock_data['20_day_roll_vol'] = stock_data['Volume'].rolling(20).mean()

stock_data['60-20_day_roll_vol'] = stock_data['Volume'].rolling(40).mean().shift(20)


stock_data['volume_rate_of_change'] = (stock_data['20_day_roll_vol'] / stock_data['60-20_day_roll_vol']) 

print(stock_data.head(100).reset_index()[['volume_rate_of_change','20_day_roll_vol','60-20_day_roll_vol','Volume']])
print(stock_data.head(100).reset_index())

    volume_rate_of_change  20_day_roll_vol  60-20_day_roll_vol     Volume
0                     NaN              NaN                 NaN  105260800
1                     NaN              NaN                 NaN   96034000
2                     NaN              NaN                 NaN  109938000
3                     NaN              NaN                 NaN   74641600
4                     NaN              NaN                 NaN  103472800
..                    ...              ...                 ...        ...
95               0.890179      122610920.0         137737440.0  104343600
96               0.910340      123165640.0         135296330.0   56998000
97               0.903320      121386220.0         134377820.0   73187600
98               0.887466      119861640.0         135060520.0   83623600
99               0.852133      115637080.0         135703070.0   60158000

[100 rows x 4 columns]
         Date      Close       High        Low       Open     Volume  daily_pct_change  

In [216]:
# Feature 2

# 20 day price change : Close price day 1 - Close price of day 20


stock_data['20_day_price_change'] = stock_data['Close'].pct_change(periods=20)*100

# print(stock_data.head(30).reset_index()[['Close','20_day_price_change']])

stock_data['positive_days_20d'] = (stock_data['daily_pct_change'] > 0).rolling(20).sum()

# print(stock_data.head(30).reset_index()[['To_be_counted','daily_pct_change','positive_days_20d']])



### Feature 2: price_change_20d

**Definition**

    (close today - close 20 days ago) / close 20 days ago  x  100


**Why I chose it**

This is the core of the hypothesis. If a stock has been trending over 20 days, I'm
testing whether that trend continues into the next 5. This feature *is* the trend —
where the price started, where it ended.

**Why point-to-point rather than averaging the daily changes**

My first instinct was to average the 20 daily percentage changes. That has a built-in
upward bias.

A price that goes 100 -> 110 -> 100 -> 110 and ends exactly where it started produces
daily changes of +10%, -9.09%, +10%, -9.09%... which average to **+0.455%**. It reports
an uptrend where nothing happened. The cause is that percentage changes are asymmetric:
going up 10% and coming back down is only -9.09%, so oscillation leaves a fake positive
residue.

Worse, the size of that bias grows with volatility — so choppy windows would be inflated
more than calm ones. That's a confound baked directly into the feature.

Point-to-point has none of this. Where did it start, where did it end. Nothing accumulates.


**What it doesn't capture**

The *path*. A steady climb and a flat month ending in one huge jump produce the same
number. That's what feature 3 is for.

---

### Feature 3: positive_days_20d

**Definition**

Count of days in the last 20 where the daily return was positive. Range 0 to 20.

In code: `(daily_pct_change > 0).rolling(20).sum()`

**Why I chose it**

Feature 2 tells me the net move but nothing about how it happened. A +5% month could be
17 red days rescued by three enormous green ones — or a steady grind upward. Those look
identical to `price_change_20d` and feel completely different as a trend.

This feature measures **consistency**: how often the stock actually went up, ignoring size
entirely. Together the two describe both the *magnitude* and the *reliability* of the move.

**The multicollinearity question**

Measured on the actual data:

| pair | correlation |
|---|---|
| price_change_20d vs positive_days_20d | **0.736** |
| price_change_20d vs volume_rate_of_change | -0.201 |
| positive_days_20d vs volume_rate_of_change | -0.122 |

Volume is essentially independent of the other two — good, it's carrying its own
information. The price/consistency pair is correlated, as expected: a stock that gained
5% probably did have more up days.

Correlated features are a problem because the model struggles to attribute credit between
them. Many different weight splits fit the data almost equally well, so it breaks the tie
using noise rather than signal — producing weights that swing between samples and can't be
read as "this feature matters more than that one".

**Quantifying it: Variance Inflation Factor**

    VIF = 1 / (1 - r^2)          standard errors get multiplied by sqrt(VIF)

| r | VIF | SE multiplier |
|---|---|---|
| 0.000 | 1.00 | 1.00x |
| **0.736 (mine)** | **2.18** | **1.48x** |
| 0.900 | 5.26 | 2.29x |
| 0.990 | 50.25 | 7.09x |

So my weight standard errors are inflated by about **1.48x** compared to uncorrelated
features. The conventional thresholds are: VIF under 5 is fine, 5-10 deserves attention,
above 10 is a real problem. **At 2.18 I'm comfortably in the acceptable range.**

**Why I'm keeping both features anyway**

*The disagreement cases are real and they're exactly what I care about.* Simulating 3000
windows turned up genuine examples: +4.66% net gain with only 8 up days out of 20, and
-4.02% net loss with 12 up days out of 20. Those are the "few big days carrying a
mostly-red month" scenarios that motivated the feature. Drop one and the model can't see
them at all.

*And 2.18 is mild.* The genuinely broken cases sit near r=0.99, where VIF hits 50 and one
sample gives [0.47, 1.55] while another gives [1.33, 0.66] for the same underlying truth.
Nowhere near that here.

In [217]:
# Feature 4 avg_gap_20_days

stock_data['daily_gap'] = ((stock_data['Open'] - stock_data['Close'].shift(1))
                           / stock_data['Close'].shift(1)) * 100



stock_data['avg_gap_20d'] = stock_data['daily_gap'].rolling(20).mean()

# print(stock_data.head(30).reset_index()[['Close', 'Open', 'daily_gap','avg_gap_20d']])


# Feature 5 gap_up_count_20d


is_gap_up = (stock_data['daily_gap'] > 0).where(stock_data['daily_gap'].notna())
stock_data['gap_up_count_20d'] = is_gap_up.rolling(20).sum()

corr = stock_data[['20_day_price_change', 'positive_days_20d',
                   'volume_rate_of_change', 'gap_up_count_20d',
                   'avg_gap_20d']].corr()

corr.round(3).style.background_gradient(cmap='coolwarm', vmin=-1, vmax=1)
# print(stock_data[['20_day_price_change', 'positive_days_20d','volume_rate_of_change','gap_up_count_20d','avg_gap_20d']].corr())

,20_day_price_change,positive_days_20d,volume_rate_of_change,gap_up_count_20d,avg_gap_20d
20_day_price_change,1.000000,0.736000,-0.201000,0.394000,0.646000
positive_days_20d,0.736000,1.000000,-0.122000,0.333000,0.415000
volume_rate_of_change,-0.201000,-0.122000,1.000000,-0.010000,-0.251000
gap_up_count_20d,0.394000,0.333000,-0.010000,1.000000,0.629000
avg_gap_20d,0.646000,0.415000,-0.251000,0.629000,1.000000


### Feature 4: avg_gap_20d

**Definition**

    daily_gap = (today's Open - yesterday's Close) / yesterday's Close x 100
    avg_gap_20d = mean of the last 20 daily_gap values

Signed: positive gaps and negative gaps both included, so the sign shows net overnight
direction and the magnitude shows how strong it was.

**Why I chose it**

Markets are shut overnight, so news, earnings and overseas moves accumulate and the price
can open somewhere quite different from where it closed. That overnight jump is a distinct
kind of pressure from intraday drift — it's where information arrives all at once.

If a 20-day trend is genuine, I'd expect it to show up in how the stock keeps *opening*,
not just how it closes.

**Why the mean and not the median**

The mean is outlier-sensitive. I tested this: dropping one -8% earnings gap into an
otherwise mild window flipped the mean from **+0.350 to -0.048** — a sign flip caused by a
single day. The median stayed at +0.245, completely unmoved.

So the median is the robust choice, and I nearly used it. I didn't, because the median
throws away exactly the events I care about. Two windows can have identical medians while
one contains an 8% overnight surge — real news, real information — and the median simply
cannot see it.

There's no measure that keeps magnitude *and* resists outliers, because those are the same
property. Sensitivity to large values is what makes something outlier-vulnerable.

The question is really: **is a big gap signal or noise?** My hypothesis says signal — an
8% earnings pop is the strongest possible version of "overnight pressure supporting a
trend". So I kept the mean, and added a count feature to cover its blind spot.

**What it doesn't capture**

Consistency. One huge gap and twenty small ones can produce similar averages.
That's what feature 5 is for.

---

### Feature 5: gap_up_count_20d

**Definition**

Count of days in the last 20 where the gap was positive. Range 0 to 20.

    is_gap_up = (daily_gap > 0).where(daily_gap.notna())
    gap_up_count_20d = is_gap_up.rolling(20).sum()

**Why I chose it**

It covers feature 4's weakness. `avg_gap_20d` sees magnitude but can be flipped by a
single day; this counts *how often* the stock opened higher and is completely immune to
size. Together, if the mean says "gapping up" but the count says "only 6 of 20 days", I
know one outlier is driving the average.

They cover each other's blind spots — magnitude and consistency.

**Why the NaN handling matters**

`daily_gap` has one NaN (the first row, no previous close). Writing `daily_gap > 0` would
silently turn that NaN into **False** — recording "this was not a gap up" when the truth is
"I don't know". The count would then be built partly on a fabricated observation.

`.where(daily_gap.notna())` keeps the NaN as NaN, so any rolling window containing it
correctly refuses to produce a count.

In this dataset it happens not to change anything, because the volume feature needs 60 days
of history and those early rows get dropped anyway. I fixed it regardless — the dataset
should represent what is known, not paper over a gap. And if I ever removed the volume
feature, this bug would silently activate.

---

### Correlation between all five features

|  | price_chg | pos_days | volume | gap_count | avg_gap |
|---|---|---|---|---|---|
| **price_change_20d** | 1.000 | 0.736 | -0.201 | 0.394 | 0.646 |
| **positive_days_20d** | 0.736 | 1.000 | -0.122 | 0.333 | 0.415 |
| **volume_rate_of_change** | -0.201 | -0.122 | 1.000 | -0.010 | -0.251 |
| **gap_up_count_20d** | 0.394 | 0.333 | -0.010 | 1.000 | 0.629 |
| **avg_gap_20d** | 0.646 | 0.415 | -0.251 | 0.629 | 1.000 |

**What this says**

- **Volume is essentially independent of everything** (-0.25 to -0.01). It's carrying its
  own information, which is exactly what I wanted from it.
- The **strongest pair is price_change / positive_days at 0.736** — expected, they're two
  views of the same 20-day move.
- **avg_gap correlates 0.646 with price_change and 0.629 with gap_up_count** — moderate.
  Overnight moves are part of the overall move, so some overlap is unavoidable.
- **gap_up_count vs positive_days is only 0.333** — usefully low. Overnight direction and
  full-day direction are genuinely different things; a stock can gap up and then fall all
  day.

**Proper VIF (not just pairwise)**

With five features, pairwise correlation understates the problem, because a feature can be
predicted by a *combination* of the others even when no single pairing looks high. The
correct VIF regresses each feature against all the rest.

| feature | VIF | SE multiplier |
|---|---|---|
| price_change_20d | 3.19 | 1.79x |
| positive_days_20d | 2.27 | 1.51x |
| volume_rate_of_change | 1.11 | 1.05x |
| gap_up_count_20d | 1.76 | 1.33x |
| avg_gap_20d | 2.63 | 1.62x |

Conventional thresholds: under 5 is fine, 5-10 deserves attention, above 10 is a real
problem. **Everything here is under 3.2.** The condition number of the correlation matrix
is 14.9, also comfortably below the usual concern level of 30.

So the weight standard errors are inflated by roughly 1.3x to 1.8x compared to perfectly
independent features. Real, but well within acceptable range. I'll still train with and
without Ridge and compare weight stability across walk-forward folds, since that gives me
evidence rather than an assertion.

In [228]:

# Quick inspection of the feature vectors


features_col = ['20_day_price_change', 'positive_days_20d',
                   'volume_rate_of_change', 'gap_up_count_20d',
                   'avg_gap_20d'];

# Now buidling the target column

# label positive 1 negatove 0

future_close = stock_data['Close'].shift(-5)
stock_data['future_direction_5d'] = (future_close - stock_data['Close'] > 0).where(future_close.notna())


# Adding a new column tatget_2 for price change 

stock_data['future_return_5d'] = ((future_close - stock_data['Close'])/stock_data['Close']) * 100



# print(stock_data.head(30).reset_index()[['Close', 'future_return_5d', 'future_direction_5d']])

print("share positive:", stock_data['future_direction_5d'].mean().round(3))
print(stock_data['future_return_5d'].describe().round(2))



share positive: 0.586
count    2508.00
mean        0.58
std         3.90
min       -22.75
25%        -1.63
50%         0.74
75%         2.84
max        18.41
Name: future_return_5d, dtype: float64


In [238]:
# Preapring data for the model


# First combinign the data

combined_data =  stock_data[features_col+['future_return_5d','future_direction_5d']].dropna();
# Now split

input_data = combined_data[features];
y = combined_data['future_return_5d']
y_return = combined_data['future_direction_5d'] # for the regression later


print('Do the index match ?', input_data.index.equals(y.index))
print('Do the index match ?', input_data.index.equals(y_return.index))

Do the index match ? True
Do the index match ? True


In [255]:
# Starting from psotion 0 and then takiiung a step of 5 each time

stepped_data = combined_data.iloc[::5]


targets = stepped_data['future_return_5d']
print("neighbour correlation:", round(targets.autocorr(lag=1), 3))

input_stepped_data = input_data.iloc[::5]


neighbour correlation: -0.051


### Stepping every 5 days: why the row count drops from 2449 to 490

**The problem**

My target is "what happened over the next 5 days". With one row per trading day,
consecutive rows describe almost the same stretch of time:

    row 1  ->  target covers days  2, 3, 4, 5, 6
    row 2  ->  target covers days     3, 4, 5, 6, 7
    row 3  ->  target covers days        4, 5, 6, 7, 8

Rows 1 and 2 **share 4 of their 5 days**. They are not two separate observations —
they are largely the same fact written down twice.

**Why that breaks the statistics**

The standard error formula is:

    SE = s / sqrt(n)

That formula assumes every row is an independent piece of evidence. It has no way to
know that rows 1 and 2 are mostly the same thing. It just counts rows.

So if I hand it 2449 rows containing only ~490 rows' worth of real information, it
believes it has five times more evidence than it does — and reports far more confidence
than it has earned.

Concretely, with 5-fold overlap:

    what the formula computes  =  s / sqrt(2449)  =  s / 49.5
    what is actually true      =  s / sqrt(490)   =  s / 22.1

    ratio = 49.5 / 22.1 = 2.24 = sqrt(5)

**The standard error comes out about 2.24x too small.** Confidence intervals are 2.24x
too narrow, and p-values are correspondingly too small.

**I verified this by simulation.** Running the same study 2000 times on data with a
*known* zero edge:

| | rows | real wobble of the answer | what the formula claimed |
|---|---|---|---|
| overlapping | 2495 | 0.1798 | 0.0809 (**2.22x too optimistic**) |
| stepped | 499 | 0.1854 | 0.1865 (honest) |

The important detail: **the real wobble is nearly identical in both** (0.18 vs 0.19).
Overlapping gave 5x the rows but the *same actual precision* — of course it did, both
used the same underlying history. The extra rows added no information. Only the formula
was fooled.

In a separate test on a strategy I built with **exactly zero edge**, the overlapping
version produced p = 0.0009 and an interval excluding zero — a confident, completely
false discovery. The stepped version correctly returned p = 0.137 and declined to claim
anything.

**The fix**

    input_stepped_data = input_data.iloc[::5]

Keep every 5th row: rows 0, 5, 10, 15... Each target now begins exactly where the previous
one ended, so no two share a day.

Rows: **2449 -> 490**

---

### Neighbour correlation, and why -0.051 is the number that matters

**What it measures**

Neighbour correlation (autocorrelation at lag 1) asks:

> *Does one row tell me anything about the next row?*

It takes each column, lines it up against itself shifted by one position, and computes
the correlation. Near zero means each row is fresh information. High means rows are
echoing each other.

This is the direct test of whether the stepping worked.

**My result**

    future_return_5d neighbour correlation:  -0.051

Before stepping this was around **+0.80**. Now it is essentially zero.

That is exactly what I wanted. Consecutive targets no longer share any days, so knowing
one week's return tells me nothing about the next. **The rows are now genuinely
independent observations**, which is precisely what the standard error formula assumes.

(-0.051 rather than exactly 0 is just sampling noise across ~490 rows. Nothing meaningful.)

**An important thing this does NOT change**

The correlations *between features* barely moved — for example price_change vs
positive_days went from 0.738 to 0.730.

**That is correct and expected.** Those two correlations measure completely different things:

| | measures | before | after | should it change? |
|---|---|---|---|---|
| feature vs feature | do two features move together? | 0.738 | 0.730 | **no** — it's a real relationship |
| row vs next row | are observations independent? | ~0.80 | **-0.051** | **yes** — this is the whole point |

Stepping was never about the feature correlations. Trending stocks genuinely do have more
up days; subsampling does not alter a real relationship. Stepping was about making the
*rows* independent so the significance tests are honest. The -0.051 confirms that worked.


---

### What this costs

Going from 2449 rows to 490 feels like discarding 80% of the data. It isn't — I'm
discarding 80% of the *rows* while keeping essentially all of the *information*, since the
dropped rows were near-duplicates. The simulation above confirms this: real precision
barely changed.

What I actually give up is false confidence. My final confidence intervals will be roughly
2.2x wider than the overlapping version would have shown. Same evidence, honestly reported.

**One arbitrary choice, declared upfront:** I start at row 0. Starting at row 1, 2, 3 or 4
would give an equally valid set of ~490 non-overlapping rows with slightly different
results. I picked row 0 before looking at any outcome and am not going to try the others
and keep whichever looks best — that would be exactly the peeking this whole exercise is
designed to prevent.